In [1]:
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import UTILS.utils as utils
import pandas as pd

import numpy as np
import seaborn as sns

import math
from umap import UMAP
import plotly.express as px

ImportError: cannot import name 'UMAP' from 'umap' (/opt/anaconda3/lib/python3.12/site-packages/umap/__init__.py)

In [4]:
t = pd.read_csv("../../Data/ALL_IMAGES_CLASSED.csv", index_col=0)

In [5]:
t

,categories,categories_probs,length_cats
0,"['Harbor and Waterways', 'British Colonialism'...","[0.13306877541416887, 0.13306860093450754, 0.1...",5
1,"['Harbor and Waterways', 'Geography and Topogr...","[0.14292729226385525, 0.14292724114284192, 0.1...",5
2,"['Harbor and Waterways', 'Daily Life and Occup...","[0.1114466961304801, 0.11144668284337765, 0.11...",5
3,"['Harbor and Waterways', 'Daily Life and Occup...","[0.14110143282833892, 0.14109993592288197, 0.1...",5
4,"['Architecture and Building', 'Geography and T...","[0.49999085101228186, 0.4999624808437444]",2
...,...,...,...
95,['Architecture and Building'],[0.9998826049821822],1
96,"['Social Classes and Customs', 'Transportation...","[0.17309618859446635, 0.1730961060553289, 0.17...",5
97,"['Photography and Imaging', 'War and Conflict'...","[0.20176117883387865, 0.20175882189001632, 0.2...",5
98,"['Photography and Imaging', 'Social Classes an...","[0.29518242591817284, 0.2951726443010741, 0.29...",4


In [21]:
all_sentences = pd.read_csv("../../Data/ALL_SENTENCES_CLASSED.csv")
topics = utils.get_list_topics()

In [13]:
import google.generativeai as genai

genai.configure(api_key=utils.API_KEY())
model = genai.GenerativeModel('gemini-1.5-flash')

In [14]:
def create_probability_distribution_prompt(document, topic):
    prompt = "Your task will be to judge whether the following document discusses the given topic."
    prompt += "\nDOCUMENT : {" + document + "}\nTOPIC : " + topic +  "\n"
    prompt += """
    Your response should be of the form 'Y' if you believe that the document belongs to the topic, and 'N' otherwise. Please provide no other commentary in your response."""
    return prompt

In [ ]:
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_prob_distribution_all_topics(sentences):
    all_rows = []
    beginning_index = 0
    while beginning_index < 2200:
        for i, sentence in tqdm(enumerate(sentences[beginning_index:beginning_index+100])):
            this_row = []
            for topic in topics:
                prompt = create_probability_distribution_prompt(sentence, topic)
                response = model.generate_content(contents=[prompt])
                logprob = response.candidates[0].avg_logprobs
                if response.text[0] == "N":
                    logprob = 1000 + logprob
                elif response.text[0] != "Y":
                    print(response.text)
                    raise SyntaxError
                this_row.append(logprob)
            all_rows.append(this_row)
        df = pd.DataFrame(all_rows, columns = topics)
        end_index = beginning_index + 100
        df_name = "TOPIC_PROBS/ROWS_" + str(beginning_index) + "_" + str(end_index) + ".csv"
        df.to_csv(df_name)  
        beginning_index += 100
        all_rows = []
    return all_rows
            


In [57]:
all_rows = get_prob_distribution_all_topics(all_sentences["text"].to_list())

53it [08:31,  9.65s/it]


In [4]:
all_csvs = utils.get_files_in_directory("TOPIC_PROBS/")
topic_probs = []
for csv in all_csvs:
    topic_probs.append(pd.read_csv(csv))
topic_probs_final = pd.concat(topic_probs)

In [ ]:
import math

## calculate true probability from log prob
def get_prob_from_log_prob(x):
    inverse = False
    if x > 100:
        x -= 1000
        inverse = True
    p = math.exp(2*x)
    if(inverse):
        p = 1 - p
    return p

func = np.vectorize(get_prob_from_log_prob)
res = func(topic_probs_final.to_numpy())

probs = pd.DataFrame(data=res, columns = topic_probs_final.columns)
probs.to_csv("../../Data/TOPIC_MODELLING/PROBABILITY_DISTRIBUTION_SENTENCES.csv", index=False)